# Specimen 02 — Shared Store Under Concurrency

Goal: reuse Phase 5's `ResearchStore`, but now under real concurrency. See a race condition actually happen, then fix it with `asyncio.Lock` -- the concurrency bug Phase 5 deliberately deferred until now.

In [1]:
import os
import json
import time
import asyncio
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.AsyncAnthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'

INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

# Caps how many requests are in flight at once, regardless of how many coroutines are scheduled --
# asyncio.gather alone fires every call simultaneously, which is the fastest way to get rate-limited.
_concurrency_limit = asyncio.Semaphore(5)

async def call_model(messages, tools=None, max_tokens=1200, output_schema=None):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages, thinking={"type": "disabled"})
    if tools:
        kwargs['tools'] = tools
    if output_schema:
        kwargs['output_config'] = {"format": {"type": "json_schema", "schema": output_schema}}
    async with _concurrency_limit:
        return await client.messages.create(**kwargs)


`call_model` is now `async` and built on `AsyncAnthropic` -- `asyncio.gather` needs real awaitable coroutines to get genuine concurrency; wrapping the sync client wouldn't actually overlap requests. A `Semaphore` caps in-flight requests at 5, since `gather` alone will fire every call at once. Everything else carries forward from Phase 3-5: `thinking` disabled by default, schema-enforced JSON handoffs via `output_schema`, and a default `max_tokens` of 1200 -- Phase 5 found 800 too tight for planner/researcher-style structured output, where the model kept writing past the budget and breaking the JSON mid-generation. One more Phase 5 finding worth remembering here: `output_config`'s JSON schema does not support `maxItems` on arrays -- bound response length through the prompt (ask for an exact count, not "at most"), not the schema.

## 1. Reuse the unlocked `ResearchStore` from Phase 5 Specimen 04

Same `get`/`set` dict wrapper, unchanged. It was already shaped for this moment -- the point of building it as its own class back in Phase 5 was so this step could be a lock addition, not a rewrite.

In [2]:
class ResearchStore:
    def __init__(self):
        self._data = {}

    def set(self, key, value):
        self._data[key] = value

    def get(self, key):
        return self._data.get(key)

    def items(self):
        return list(self._data.items())

    def __len__(self):
        return len(self._data)

print("Unlocked ResearchStore ready (same shape as Phase 5) -- about to prove why it isn't safe as-is.")

Unlocked ResearchStore ready (same shape as Phase 5) -- about to prove why it isn't safe as-is.


## 2. Deliberately trigger a lost update

Don't rely on real API-call timing to force a race -- it's unreliable. Instead, have several coroutines read-modify-write the same store entry (e.g. incrementing a shared counter) with an `await asyncio.sleep(0)` between the read and the write, run them concurrently, and show the final value is *wrong* -- lower than the number of increments performed.

In [3]:
class UnsafeCounter:
    def __init__(self):
        self.value = 0

    async def increment(self):
        current = self.value
        await asyncio.sleep(0)  # yields control -- simulates the gap between read and write under real concurrency
        self.value = current + 1

counter = UnsafeCounter()
N = 50
await asyncio.gather(*[counter.increment() for _ in range(N)])

print(f"Expected value: {N}")
print(f"Actual value:   {counter.value}")
print(f"Lost updates:   {N - counter.value}")

Expected value: 50
Actual value:   1
Lost updates:   49


## 3. Wrap `get`/`set` in an `asyncio.Lock`, and confirm the race is gone

Rerun the exact same lost-update scenario from step 2 against the locked store. The final value should now be exactly correct, every time.

In [4]:
class SafeCounter:
    def __init__(self):
        self.value = 0
        self._lock = asyncio.Lock()

    async def increment(self):
        async with self._lock:
            current = self.value
            await asyncio.sleep(0)
            self.value = current + 1

safe_counter = SafeCounter()
await asyncio.gather(*[safe_counter.increment() for _ in range(N)])

print(f"Expected value: {N}")
print(f"Actual value:   {safe_counter.value}")
print(f"Lost updates:   {N - safe_counter.value}")

Expected value: 50
Actual value:   50
Lost updates:   0


## 4. Apply the locked store to real concurrent agent writes

Multiple researcher agents (real API calls) writing their results into the store at once, using `asyncio.gather`.

In [5]:
class LockedResearchStore:
    def __init__(self):
        self._data = {}
        self._lock = asyncio.Lock()

    async def set(self, key, value):
        async with self._lock:
            self._data[key] = value

    async def get(self, key):
        async with self._lock:
            return self._data.get(key)

    def items(self):
        return list(self._data.items())

    def __len__(self):
        return len(self._data)

store = LockedResearchStore()

topics = {
    "topic_telephone": "the invention of the telephone",
    "topic_penicillin": "the discovery of penicillin",
    "topic_www": "the founding of the World Wide Web",
    "topic_heart_transplant": "the first successful human heart transplant",
    "topic_transistor": "the invention of the transistor",
}

async def research_and_store(key, subject, store):
    response = await call_model(
        messages=[{"role": "user", "content": f"In exactly one sentence, state a key fact about {subject}."}],
        max_tokens=150,
    )
    fact = ''.join(b.text for b in response.content if b.type == 'text')
    await store.set(key, fact)
    return key

completed = await asyncio.gather(*[research_and_store(key, subject, store) for key, subject in topics.items()])
print(f"Dispatched {len(topics)} concurrent researchers, completed: {completed}")

Dispatched 5 concurrent researchers, completed: ['topic_telephone', 'topic_penicillin', 'topic_www', 'topic_heart_transplant', 'topic_transistor']


## 5. Confirm no entries go missing under real concurrent writes

Check the store afterward: every subtopic that was dispatched should have exactly one entry, with no overwritten or dropped results.

In [6]:
print(f"Store holds {len(store)} entries (expected {len(topics)}):")
for key, fact in sorted(store.items()):
    print(f"  {key}: {fact}")

missing = set(topics) - set(k for k, _ in store.items())
print(f"\nMissing keys: {missing if missing else 'none'}")

Store holds 5 entries (expected 5):
  topic_heart_transplant: The first successful human heart transplant was performed by surgeon Christiaan Barnard on December 3, 1967, at Groote Schuur Hospital in Cape Town, South Africa, on patient Louis Washkansky, who survived 18 days before dying of pneumonia.
  topic_penicillin: Penicillin was discovered in 1928 by Alexander Fleming, who noticed that a stray *Penicillium* mould contaminating one of his Staphylococcus culture plates had killed the surrounding bacteria.
  topic_telephone: Alexander Graham Bell was granted the first U.S. patent for the telephone on March 7, 1876, though rival inventor Elisha Gray filed a similar claim the very same day, sparking a lasting dispute over who truly invented it.
  topic_transistor: The transistor was invented in December 1947 at Bell Labs by John Bardeen, Walter Brattain, and William Shockley, who shared the 1956 Nobel Prize in Physics for the achievement.
  topic_www: The World Wide Web was invented b

## 6. Name the tradeoff you just accepted

This is one global lock, not a per-key lock -- concurrent writers to *different* keys still serialize on it, even though they don't actually conflict. That's a deliberate simplicity choice (per the final-project design decision to keep the store a plain locked dict, not a database) -- write a couple of sentences on when that tradeoff would start to matter.

In [7]:
reflection = """
This store uses one global asyncio.Lock, not a per-key lock. topic_telephone's write and
topic_penicillin's write touch completely different dict keys and can never actually conflict,
but they still queue behind each other on the same lock -- the store adds a small serialization
cost that isn't strictly necessary for cross-key writes.

For a handful of fast local dict operations this cost is invisible next to the seconds-long API
calls that dominate the runtime. It would start to matter if the store itself became the
bottleneck -- e.g. hundreds of workers writing at very high frequency to unrelated keys, where
contention on one global lock could actually throttle throughput. The fix at that point would be
a per-key lock (a dict of asyncio.Lock objects, created on first access per key) -- more complex,
and not worth building until the simple version is actually measured to be slow.
"""
print(reflection.strip())

This store uses one global asyncio.Lock, not a per-key lock. topic_telephone's write and
topic_penicillin's write touch completely different dict keys and can never actually conflict,
but they still queue behind each other on the same lock -- the store adds a small serialization
cost that isn't strictly necessary for cross-key writes.

For a handful of fast local dict operations this cost is invisible next to the seconds-long API
calls that dominate the runtime. It would start to matter if the store itself became the
bottleneck -- e.g. hundreds of workers writing at very high frequency to unrelated keys, where
contention on one global lock could actually throttle throughput. The fix at that point would be
a per-key lock (a dict of asyncio.Lock objects, created on first access per key) -- more complex,
and not worth building until the simple version is actually measured to be slow.
